In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
from ollama import Client
from IPython.display import Audio
from kokoro import KPipeline
import soundfile as sf
import re
from qdrant_client import QdrantClient
from io import BytesIO
from pydub import AudioSegment
import sys
import os
import language_tool_python
import re

sys.path.append(os.path.abspath('../Backend'))

from generate_meditation import generate_text, parsear_meditacion, ajustar_silencios_y_reconstruir
# Configuration


In [3]:
tool = language_tool_python.LanguageTool('es')

In [11]:
texto = generate_text(4, "principiante")

In [12]:
segmentos = parsear_meditacion(texto)
pipeline = KPipeline(lang_code='e')

In [14]:
segmentos

[('texto',
  'Sentaos con comodidad y cierra los ojos. Toma una respiración profunda...'),
 ('silencio', 10),
 ('texto',
  'Ahora, concéntrate en la respiración, siente cómo el aire entra y sale por la nariz. No pienses en nada más, solo la respiración.'),
 ('silencio', 15),
 ('texto',
  'Siente el peso del cuerpo sobre la silla o el suelo. La sensación de los pies tocando el suelo, las manos descansando en los muslos...'),
 ('silencio', 10),
 ('texto',
  'Ahora, concéntrate en tus sentidos. Siente cómo se sienten las palpitaciones del corazón, la humedad en la nariz y los labios. El sonido de tu respiración...'),
 ('silencio', 15),
 ('texto',
  'La mente es inquieta, pensamientos vienen y vano. Sin juicio ni apego, observa estos pensamientos como nubes que pasan por el cielo.'),
 ('silencio', 20),
 ('texto',
  'Recuerda la permanencia de todo lo que existe. Nada permanece igual, todo cambia constantemente. Incluso tú mismo eres un proceso de cambio y transformación.'),
 ('silencio', 3

In [6]:
def generar_audio_segmentos(segmentos, pipeline):
    audio_texto = []
    dur_texto_muestras = 0
    dur_silencios_segundos = 0
    for tipo, contenido in segmentos:
        if tipo == 'texto':
            #logger.info(f"Generando audio para texto: {contenido}")
            generator = pipeline(contenido, voice="ef_dora", speed=0.8)
            audio = np.concatenate([audio for _, _, audio in generator])
            audio_texto.append(audio)
            dur_texto_muestras += len(audio)
        elif tipo == 'silencio':
            dur_silencios_segundos += contenido
    return audio_texto, dur_texto_muestras, dur_silencios_segundos

In [7]:
audio_texto, dur_texto_muestras, dur_silencios_segundos = generar_audio_segmentos(segmentos, pipeline)

In [8]:
audio_final = ajustar_silencios_y_reconstruir(segmentos, audio_texto, dur_texto_muestras, dur_silencios_segundos, 5 * 60)

In [9]:
output_path = f"../data/audio/meditacion_kokoro_test.wav"
sf.write(output_path, audio_final, samplerate=24000)